In [ ]:
# =========================================================
# IMPORTS
# =========================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os

from tqdm import tqdm
from transformers import T5Tokenizer, T5ForSequenceClassification
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import autocast, GradScaler


In [ ]:
# =========================================================
# LOAD DATASET
# =========================================================
df = pd.read_csv("/content/drive/MyDrive/multilabel-classification/dataset/train.csv")

df["text"] = df["TITLE"].astype(str) + " " + df["ABSTRACT"].astype(str)

LABEL_NAMES = [
    "Computer Science",
    "Physics",
    "Mathematics",
    "Statistics",
    "Quantitative Biology",
    "Quantitative Finance"
]

labels = df[LABEL_NAMES].values
texts = df["text"].tolist()

print("Dataset size:", len(df))

In [ ]:
# =========================================================
# TOKENIZER
# =========================================================
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)

In [ ]:
# =========================================================
# DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# =========================================================
# TRAINING SETTINGS
# =========================================================
num_epochs = 3
batch_size = 4

criterion = nn.BCEWithLogitsLoss()

scaler = GradScaler()

checkpoint_dir = "/content/drive/MyDrive/multilabel-classification/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)


In [ ]:
# =========================================================
# K-FOLD SETUP
# =========================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
# =========================================================
# STORE FOLD RESULTS
# =========================================================

fold_results = {
    "Subset Accuracy": [],
    "Label Accuracy": [],
    "Precision": [],
    "Recall": [],
    "F1 Micro": [],
    "F1 Macro": [],
    "Val Loss": []
}

# =========================================================
# TRAINING LOOP
# =========================================================
for fold, (train_idx, val_idx) in enumerate(kf.split(texts)):

    print("\n==================================================")
    print(f"Starting Fold {fold+1}")
    print("==================================================")

    train_texts = [texts[i] for i in train_idx]
    val_texts = [texts[i] for i in val_idx]

    train_labels = labels[train_idx]
    val_labels = labels[val_idx]

    # =========================================================
    # TOKENIZATION
    # =========================================================
    train_encodings = tokenizer(
        train_texts,
        truncation=True,
        padding=True,
        max_length=512
    )

    val_encodings = tokenizer(
        val_texts,
        truncation=True,
        padding=True,
        max_length=512
    )

    # =========================================================
    # DATASETS
    # =========================================================
    train_dataset = TensorDataset(
        torch.tensor(train_encodings["input_ids"]),
        torch.tensor(train_encodings["attention_mask"]),
        torch.tensor(train_labels, dtype=torch.float32)
    )

    val_dataset = TensorDataset(
        torch.tensor(val_encodings["input_ids"]),
        torch.tensor(val_encodings["attention_mask"]),
        torch.tensor(val_labels, dtype=torch.float32)
    )

    # =========================================================
    # DATALOADERS
    # =========================================================
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        pin_memory=True
    )

    # =========================================================
    # MODEL RESET EACH FOLD
    # =========================================================
    model = T5ForSequenceClassification.from_pretrained(model_name, num_labels=6)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    # =========================================================
    # EPOCH LOOP
    # =========================================================
    for epoch in range(num_epochs):

        print(f"\nEpoch {epoch+1}/{num_epochs} Fold {fold+1}")
        print("==================================================")

        # ================= TRAIN =================
        model.train()

        total_train_loss = 0
        train_preds, train_true = [], []

        for batch in tqdm(train_loader):

            optimizer.zero_grad()

            input_ids, attention_mask, labels_batch = [b.to(device) for b in batch]

            with autocast():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                logits = outputs.logits
                loss = criterion(logits, labels_batch)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss.item()

            preds = torch.sigmoid(logits).detach().cpu().numpy()

            train_preds.append(preds)
            train_true.append(labels_batch.cpu().numpy())

        train_preds = np.vstack(train_preds)
        train_true = np.vstack(train_true)

        train_bin = (train_preds > 0.5).astype(int)

        train_loss = total_train_loss / len(train_loader)

        train_subset_acc = accuracy_score(train_true, train_bin)
        train_label_acc = (train_true == train_bin).mean()

        train_prec = precision_score(train_true, train_bin, average="micro", zero_division=0)
        train_rec = recall_score(train_true, train_bin, average="micro", zero_division=0)

        train_f1_micro = f1_score(train_true, train_bin, average="micro", zero_division=0)
        train_f1_macro = f1_score(train_true, train_bin, average="macro", zero_division=0)

        print("TRAIN METRICS")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Subset Accuracy: {train_subset_acc:.4f}")
        print(f"Label Accuracy: {train_label_acc:.4f}")
        print(f"Precision: {train_prec:.4f}")
        print(f"Recall: {train_rec:.4f}")
        print(f"F1 Micro: {train_f1_micro:.4f}")
        print(f"F1 Macro: {train_f1_macro:.4f}")

        # ================= VALIDATION =================
        model.eval()

        total_val_loss = 0
        val_preds, val_true = [], []

        with torch.no_grad():

            for batch in tqdm(val_loader):

                input_ids, attention_mask, labels_batch = [b.to(device) for b in batch]

                with autocast():
                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask
                    )

                    logits = outputs.logits
                    loss = criterion(logits, labels_batch)

                total_val_loss += loss.item()

                preds = torch.sigmoid(logits).cpu().numpy()

                val_preds.append(preds)
                val_true.append(labels_batch.cpu().numpy())

        val_preds = np.vstack(val_preds)
        val_true = np.vstack(val_true)

        val_bin = (val_preds > 0.5).astype(int)

        val_loss = total_val_loss / len(val_loader)

        val_subset_acc = accuracy_score(val_true, val_bin)
        val_label_acc = (val_true == val_bin).mean()

        val_prec = precision_score(val_true, val_bin, average="micro", zero_division=0)
        val_rec = recall_score(val_true, val_bin, average="micro", zero_division=0)

        val_f1_micro = f1_score(val_true, val_bin, average="micro", zero_division=0)
        val_f1_macro = f1_score(val_true, val_bin, average="macro", zero_division=0)

        print("\nVALIDATION METRICS")
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Subset Accuracy: {val_subset_acc:.4f}")
        print(f"Label Accuracy: {val_label_acc:.4f}")
        print(f"Precision: {val_prec:.4f}")
        print(f"Recall: {val_rec:.4f}")
        print(f"F1 Micro: {val_f1_micro:.4f}")
        print(f"F1 Macro: {val_f1_macro:.4f}")

        # ================= CHECKPOINT =================
        checkpoint_path = f"{checkpoint_dir}/t5_fold{fold+1}_epoch{epoch+1}.pt"

        torch.save(
            {
                "epoch": epoch,
                "fold": fold,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict()
            },
            checkpoint_path
        )

        print(f"\nCheckpoint saved → {checkpoint_path}")

        torch.cuda.empty_cache()

    # =========================================================
    # SAVE FINAL FOLD RESULTS (after last epoch)
    # =========================================================
    fold_results["Subset Accuracy"].append(val_subset_acc)
    fold_results["Label Accuracy"].append(val_label_acc)
    fold_results["Precision"].append(val_prec)
    fold_results["Recall"].append(val_rec)
    fold_results["F1 Micro"].append(val_f1_micro)
    fold_results["F1 Macro"].append(val_f1_macro)
    fold_results["Val Loss"].append(val_loss)

In [ ]:
# =========================================================
# CROSS-VALIDATION TABLE (Mean ± Std)
# =========================================================
import pandas as pd
import numpy as np

# Create DataFrame from fold_results
results_df = pd.DataFrame(fold_results)
results_df.index = [f"Fold {i+1}" for i in range(len(results_df))]

print("\n================ FOLD RESULTS ================\n")
print(results_df)

# Compute Mean and Std
mean_results = results_df.mean()
std_results = results_df.std()

# Format as "mean ± std" for table
cv_summary = pd.DataFrame({
    "CV Result (Mean ± Std)": [
        f"{mean_results['Subset Accuracy']:.4f} ± {std_results['Subset Accuracy']:.4f}",
        f"{mean_results['Label Accuracy']:.4f} ± {std_results['Label Accuracy']:.4f}",
        f"{mean_results['Precision']:.4f} ± {std_results['Precision']:.4f}",
        f"{mean_results['Recall']:.4f} ± {std_results['Recall']:.4f}",
        f"{mean_results['F1 Micro']:.4f} ± {std_results['F1 Micro']:.4f}",
        f"{mean_results['F1 Macro']:.4f} ± {std_results['F1 Macro']:.4f}",
        f"{mean_results['Val Loss']:.4f} ± {std_results['Val Loss']:.4f}"
    ]
}, index=[
    "Subset Accuracy",
    "Label Accuracy",
    "Precision",
    "Recall",
    "F1 Micro",
    "F1 Macro",
    "Val Loss"
])

print("\n================ CROSS-VALIDATION SUMMARY ================\n")
print(cv_summary)

# Optional: save to CSV
cv_summary.to_csv("/content/drive/MyDrive/multilabel-classification/working/cv_summary.csv")
results_df.to_csv("/content/drive/MyDrive/multilabel-classification/fold_results.csv")